In [ ]:
# CELL 1: SETUP AND IMPORTS
!pip install pytorch-lightning albumentations scikit-learn h5py tqdm opencv-python timm torchmetrics

import os
import cv2
import csv
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import KFold, train_test_split
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassF1Score,
    MulticlassPrecision,
    MulticlassRecall
)

# Ensure strict reproducibility
pl.seed_everything(42, workers=True)
print(f"PyTorch          : {torch.__version__}")
print(f"Lightning        : {pl.__version__}")
print(f"Timm (Models)    : {timm.__version__}")

In [ ]:
# CELL 1.5: MOUNT DRIVE AND FAST COPY DATASET
from google.colab import drive
import os

# 1. Mount Google Drive (will prompt for authorization)
drive.mount('/content/drive')

# Define paths
drive_path = "/content/drive/MyDrive/Brain FigShare Dataset"
local_path = "/content/"

# Check if it exists on drive to prevent errors
if os.path.exists(drive_path):
    print("🚀 Initiating high-speed transfer from Drive to Local Colab Storage...")

    # 2. Use rsync for the fastest possible transfer of multiple small files
    # -a: archive mode (preserves permissions/times)
    # -h: human-readable numbers
    # --info=progress2: shows overall progress rather than per-file spam
    !rsync -ah --info=progress2 "{drive_path}" "{local_path}"

    print("\n✅ Transfer complete! Dataset is now in local high-speed storage.")
else:
    print(f"❌ Error: Could not find the folder at '{drive_path}'. Please check the exact folder name in your Google Drive.")

In [ ]:
# CELL 2: EXTRACT .MAT FILES TO PNG AND CREATE LABELS CSV
source_dir = '/content/Brain FigShare Dataset/dataset/data'
output_dir = '/content/Processed_Brain_Dataset'

img_dir  = os.path.join(output_dir, 'images')
mask_dir = os.path.join(output_dir, 'masks')
os.makedirs(img_dir,  exist_ok=True)
os.makedirs(mask_dir, exist_ok=True)

csv_path  = os.path.join(output_dir, 'labels.csv')
mat_files = [f for f in os.listdir(source_dir) if f.endswith('.mat')]
label_map = {1: 'Meningioma', 2: 'Glioma', 3: 'Pituitary'}
skipped   = 0

print(f"Converting {len(mat_files)} .mat files...")

with open(csv_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    # Note: label_idx will be 0-indexed (0, 1, 2) for PyTorch CrossEntropyLoss
    writer.writerow(['filename', 'label_id', 'label_idx', 'label_name'])

    for mat_file in tqdm(mat_files):
        file_path = os.path.join(source_dir, mat_file)
        base_name = mat_file.replace('.mat', '.png')

        try:
            with h5py.File(file_path, 'r') as f:
                cjdata = f['cjdata']

                # Extract and normalize raw MRI
                image = np.array(cjdata['image']).T
                image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
                cv2.imwrite(os.path.join(img_dir, base_name), image)

                # Extract and save tumor mask
                mask = np.array(cjdata['tumorMask']).T
                mask = (mask * 255).astype(np.uint8)
                cv2.imwrite(os.path.join(mask_dir, base_name), mask)

                # Extract label and convert to 0-indexed integer
                label_id = int(cjdata['label'][0][0])
                label_idx = label_id - 1  # 1->0 (Meningioma), 2->1 (Glioma), 3->2 (Pituitary)

                writer.writerow([base_name, label_id, label_idx, label_map.get(label_id, 'Unknown')])

        except Exception as e:
            skipped += 1

print(f"Done. Converted: {len(mat_files) - skipped} | Skipped: {skipped}")

In [ ]:
# CELL 3: ROI-PATCH EXTRACTION DATASET

train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class BrainPatchDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _get_bounding_box(self, mask):
        coords = cv2.findNonZero(mask)
        if coords is None:
            return 0, 0, mask.shape[1], mask.shape[0]
        x, y, w, h = cv2.boundingRect(coords)
        return x, y, w, h

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['filename']
        label = int(row['label_idx'])

        img_path  = os.path.join(img_dir,  img_name)
        mask_path = os.path.join(mask_dir, img_name)

        # BUG 2 FIXED: Convert BGR to RGB for correct ImageNet normalization
        img  = cv2.imread(img_path,  cv2.IMREAD_COLOR)
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        x, y, w, h = self._get_bounding_box(mask)

        pad = 10
        y1 = max(0, y - pad)
        y2 = min(img.shape[0], y + h + pad)
        x1 = max(0, x - pad)
        x2 = min(img.shape[1], x + w + pad)

        tumor_patch = img[y1:y2, x1:x2]

        if self.transform:
            augmented = self.transform(image=tumor_patch)
            img_tensor = augmented['image']
        else:
            tumor_patch = cv2.resize(tumor_patch, (224, 224))
            img_tensor = torch.tensor(tumor_patch).permute(2, 0, 1).float() / 255.0

        return img_tensor, torch.tensor(label, dtype=torch.long)

In [ ]:
# CELL 4: PYTORCH LIGHTNING CLASSIFIER WITH MULTI-CLASS METRICS

class BrainTumorClassifier(pl.LightningModule):
    def __init__(self, num_classes=3, learning_rate=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate

        self.model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=num_classes)
        self.loss_fn = nn.CrossEntropyLoss()

        self.train_acc = MulticlassAccuracy(num_classes=num_classes, average='macro')

        self.val_acc   = MulticlassAccuracy(num_classes=num_classes, average='macro')
        self.val_f1    = MulticlassF1Score(num_classes=num_classes, average='macro')

        self.test_acc  = MulticlassAccuracy(num_classes=num_classes, average='macro')
        self.test_f1   = MulticlassF1Score(num_classes=num_classes, average='macro')
        self.test_prec = MulticlassPrecision(num_classes=num_classes, average='macro')
        self.test_rec  = MulticlassRecall(num_classes=num_classes, average='macro')

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)

        self.train_acc(logits, y)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_acc', self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)

        self.val_acc(logits, y)
        self.val_f1(logits, y)

        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_acc', self.val_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_f1', self.val_f1, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)

        self.test_acc(logits, y)
        self.test_f1(logits, y)
        self.test_prec(logits, y)
        self.test_rec(logits, y)

        # BUG 3 FIXED: Explicitly log test metrics so the Trainer can return them
        self.log('test_acc', self.test_acc, on_epoch=True)
        self.log('test_f1', self.test_f1, on_epoch=True)
        self.log('test_prec', self.test_prec, on_epoch=True)
        self.log('test_rec', self.test_rec, on_epoch=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.2, patience=4
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"}
        }

In [ ]:
# CELL 5: 5-FOLD CROSS VALIDATION AND EVALUATION METRICS LOOP

RESULTS_DIR = '/content/Classification_Results_EfficientNetB4'
os.makedirs(RESULTS_DIR, exist_ok=True)

df_all = pd.read_csv(csv_path)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []

print(f"Total Dataset Samples: {len(df_all)}")
print(f"Execution Split Formula: ~80% Train-Val Validation Fold / 20% Clean Test Set")

for fold, (train_val_idx, test_idx) in enumerate(kf.split(df_all), 1):
    print(f"\n{'='*65}")
    print(f"  STARTING FOLD {fold} / 5 (EFFICIENTNET-B4 CLASSIFIER)")
    print(f"{'='*65}")

    df_test = df_all.iloc[test_idx]
    df_train_val = df_all.iloc[train_val_idx]
    df_train, df_val = train_test_split(df_train_val, test_size=0.15, random_state=42)

    # BUG 1 FIXED: Updated to use BrainPatchDataset
    train_loader = DataLoader(
        BrainPatchDataset(df_train, transform=train_transform),
        batch_size=16, shuffle=True, num_workers=2, pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        BrainPatchDataset(df_val, transform=val_transform),
        batch_size=16, shuffle=False, num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        BrainPatchDataset(df_test, transform=val_transform),
        batch_size=16, shuffle=False, num_workers=2, pin_memory=True
    )

    model = BrainTumorClassifier(num_classes=3, learning_rate=2e-4)

    early_stop = EarlyStopping(monitor='val_f1', patience=8, mode='max', verbose=True)
    checkpoint = ModelCheckpoint(
        dirpath=RESULTS_DIR,
        filename=f'efficientnet_b4_fold_{fold}_best',
        save_top_k=1, monitor='val_f1', mode='max'
    )

    trainer = pl.Trainer(
        max_epochs=30,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        callbacks=[early_stop, checkpoint],
        enable_progress_bar=True,
        logger=False
    )

    trainer.fit(model, train_loader, val_loader)

    print(f"\n  Evaluating Unseen Test Data for Fold {fold}...")

    # BUG 3 FIXED: Safely capture the returned metrics dictionary
    test_results = trainer.test(model, dataloaders=test_loader, ckpt_path='best', verbose=False)

    f_acc  = test_results[0]['test_acc']
    f_f1   = test_results[0]['test_f1']
    f_prec = test_results[0]['test_prec']
    f_rec  = test_results[0]['test_rec']

    print(f"  Fold {fold} Performance Metrics:")
    print(f"  >> Accuracy  : {f_acc:.4f} | F1-Score : {f_f1:.4f}")
    print(f"  >> Precision : {f_prec:.4f} | Recall   : {f_rec:.4f}")

    fold_metrics.append([f_acc, f_f1, f_prec, f_rec])

metrics_arr = np.array(fold_metrics)
means = np.mean(metrics_arr, axis=0)
stds = np.std(metrics_arr, axis=0)

print(f"\n{'='*65}")
print(f"  OVERALL 5-FOLD BENCHMARK RESULTS SUMMARY")
print(f"{'='*65}")
print(f"  Mean Accuracy  : {means[0]:.4f} ± {stds[0]:.4f}")
print(f"  Mean F1-Score  : {means[1]:.4f} ± {stds[1]:.4f}")
print(f"  Mean Precision : {means[2]:.4f} ± {stds[2]:.4f}")
print(f"  Mean Recall    : {means[3]:.4f} ± {stds[3]:.4f}")
print(f"{'='*65}")

WITH OUR PREDICTED MASKS

In [ ]:
# CELL 1: SETUP AND IMPORTS
!pip install pytorch-lightning albumentations scikit-learn h5py tqdm opencv-python timm torchmetrics

import os
import cv2
import csv
import h5py
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import KFold, train_test_split
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassF1Score,
    MulticlassPrecision,
    MulticlassRecall
)

# Ensure strict reproducibility
pl.seed_everything(42, workers=True)
print(f"PyTorch          : {torch.__version__}")
print(f"Lightning        : {pl.__version__}")
print(f"Timm (Models)    : {timm.__version__}")

In [ ]:
# CELL 1.5: MOUNT DRIVE AND FAST COPY BOTH DATASETS
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define paths for Raw Dataset
drive_raw_path = "/content/drive/MyDrive/Brain FigShare Dataset"
local_raw_path = "/content/"

# 3. Define paths for AI-Predicted Masks (EXP2)
drive_pred_masks = "/content/drive/MyDrive/Brain_Project_Predicted_Masks"
local_pred_masks = "/content/Predicted_Masks_Local"

# 4. Copy Raw Data
if os.path.exists(drive_raw_path):
    print("🚀 Transferring Raw Data from Drive to Local Storage...")
    !rsync -ah --info=progress2 "{drive_raw_path}" "{local_raw_path}"
else:
    print(f"❌ Error: Could not find raw data at '{drive_raw_path}'.")

# 5. Copy AI-Predicted Masks Locally (Crucial for fast training speed)
if os.path.exists(drive_pred_masks):
    print("\n🚀 Transferring AI-Predicted Masks from Drive to Local Storage...")
    os.makedirs(local_pred_masks, exist_ok=True)
    # Copies the contents of the drive folder into the local folder
    !cp -a "{drive_pred_masks}/." "{local_pred_masks}/"
    print("\n✅ All transfers complete! Ready for high-speed preprocessing.")
else:
    print(f"❌ Error: Could not find predicted masks at '{drive_pred_masks}'.")

In [ ]:
# CELL 2: EXTRACT .MAT FILES TO PNG AND CREATE LABELS CSV
source_dir = '/content/Brain FigShare Dataset/dataset/data'
output_dir = '/content/Processed_Brain_Dataset'

img_dir  = os.path.join(output_dir, 'images')
os.makedirs(img_dir,  exist_ok=True)

csv_path  = os.path.join(output_dir, 'labels.csv')
mat_files = [f for f in os.listdir(source_dir) if f.endswith('.mat')]
label_map = {1: 'Meningioma', 2: 'Glioma', 3: 'Pituitary'}
skipped   = 0

print(f"Converting {len(mat_files)} .mat files...")

with open(csv_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['filename', 'label_id', 'label_idx', 'label_name'])

    for mat_file in tqdm(mat_files):
        file_path = os.path.join(source_dir, mat_file)
        base_name = mat_file.replace('.mat', '.png')

        try:
            with h5py.File(file_path, 'r') as f:
                cjdata = f['cjdata']

                # Extract and normalize raw MRI
                image = np.array(cjdata['image']).T
                image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
                cv2.imwrite(os.path.join(img_dir, base_name), image)

                # Extract label and convert to 0-indexed integer
                label_id = int(cjdata['label'][0][0])
                label_idx = label_id - 1  # 1->0, 2->1, 3->2

                writer.writerow([base_name, label_id, label_idx, label_map.get(label_id, 'Unknown')])

        except Exception as e:
            skipped += 1

print(f"Done. Converted: {len(mat_files) - skipped} | Skipped: {skipped}")

In [ ]:
# CELL 3: ROI-PATCH EXTRACTION DATASET (PROFESSOR's EXP2)

train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class BrainPatchDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform
        self.ai_masks_dir = '/content/Predicted_Masks_Local'

    def __len__(self):
        return len(self.df)

    def _get_bounding_box(self, mask):
        coords = cv2.findNonZero(mask)
        if coords is None:
            return 0, 0, mask.shape[1], mask.shape[0]
        x, y, w, h = cv2.boundingRect(coords)
        return x, y, w, h

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['filename']
        label = int(row['label_idx'])

        img_path  = os.path.join(img_dir,  img_name)
        mask_path = os.path.join(self.ai_masks_dir, img_name)

        img  = cv2.imread(img_path,  cv2.IMREAD_COLOR)
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # --- THE BUG FIX ---
        # 1. Dynamically resize the mask to match the exact image dimensions
        # This prevents out-of-bounds bounding box coordinates
        if mask.shape != img.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)

        # Get bounding box from the perfectly aligned mask
        x, y, w, h = self._get_bounding_box(mask)

        pad = 10
        y1 = max(0, y - pad)
        y2 = min(img.shape[0], y + h + pad)
        x1 = max(0, x - pad)
        x2 = min(img.shape[1], x + w + pad)

        # Crop to the AI's predicted tumor location
        tumor_patch = img[y1:y2, x1:x2]

        # 2. Failsafe: If the patch is empty for any mathematical reason, use the whole image
        if tumor_patch.size == 0:
            tumor_patch = img.copy()
        # -------------------

        if self.transform:
            augmented = self.transform(image=tumor_patch)
            img_tensor = augmented['image']
        else:
            tumor_patch = cv2.resize(tumor_patch, (224, 224))
            img_tensor = torch.tensor(tumor_patch).permute(2, 0, 1).float() / 255.0

        return img_tensor, torch.tensor(label, dtype=torch.long)

In [ ]:
# CELL 4: PYTORCH LIGHTNING CLASSIFIER WITH MULTI-CLASS METRICS

class BrainTumorClassifier(pl.LightningModule):
    def __init__(self, num_classes=3, learning_rate=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate

        self.model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=num_classes)
        self.loss_fn = nn.CrossEntropyLoss()

        self.train_acc = MulticlassAccuracy(num_classes=num_classes, average='macro')

        self.val_acc   = MulticlassAccuracy(num_classes=num_classes, average='macro')
        self.val_f1    = MulticlassF1Score(num_classes=num_classes, average='macro')

        self.test_acc  = MulticlassAccuracy(num_classes=num_classes, average='macro')
        self.test_f1   = MulticlassF1Score(num_classes=num_classes, average='macro')
        self.test_prec = MulticlassPrecision(num_classes=num_classes, average='macro')
        self.test_rec  = MulticlassRecall(num_classes=num_classes, average='macro')

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)

        self.train_acc(logits, y)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_acc', self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)

        self.val_acc(logits, y)
        self.val_f1(logits, y)

        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_acc', self.val_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_f1', self.val_f1, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)

        self.test_acc(logits, y)
        self.test_f1(logits, y)
        self.test_prec(logits, y)
        self.test_rec(logits, y)

        self.log('test_acc', self.test_acc, on_epoch=True)
        self.log('test_f1', self.test_f1, on_epoch=True)
        self.log('test_prec', self.test_prec, on_epoch=True)
        self.log('test_rec', self.test_rec, on_epoch=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.2, patience=4
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"}
        }

In [ ]:
# CELL 5: 5-FOLD CROSS VALIDATION AND EVALUATION METRICS LOOP

RESULTS_DIR = '/content/Classification_Results_EfficientNetB4_EXP2'
os.makedirs(RESULTS_DIR, exist_ok=True)

df_all = pd.read_csv(csv_path)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []

print(f"Total Dataset Samples: {len(df_all)}")
print(f"Execution Split Formula: ~80% Train-Val / 20% Clean Test Set")

for fold, (train_val_idx, test_idx) in enumerate(kf.split(df_all), 1):
    print(f"\n{'='*65}")
    print(f"  STARTING FOLD {fold} / 5 (EXP 2: AI MASKS + EFFICIENTNET-B4)")
    print(f"{'='*65}")

    df_test = df_all.iloc[test_idx]
    df_train_val = df_all.iloc[train_val_idx]
    df_train, df_val = train_test_split(df_train_val, test_size=0.15, random_state=42)

    train_loader = DataLoader(
        BrainPatchDataset(df_train, transform=train_transform),
        batch_size=16, shuffle=True, num_workers=2, pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        BrainPatchDataset(df_val, transform=val_transform),
        batch_size=16, shuffle=False, num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        BrainPatchDataset(df_test, transform=val_transform),
        batch_size=16, shuffle=False, num_workers=2, pin_memory=True
    )

    model = BrainTumorClassifier(num_classes=3, learning_rate=2e-4)

    early_stop = EarlyStopping(monitor='val_f1', patience=8, mode='max', verbose=True)
    checkpoint = ModelCheckpoint(
        dirpath=RESULTS_DIR,
        filename=f'efficientnet_b4_exp2_fold_{fold}_best',
        save_top_k=1, monitor='val_f1', mode='max'
    )

    trainer = pl.Trainer(
        max_epochs=30,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        callbacks=[early_stop, checkpoint],
        enable_progress_bar=True,
        logger=False
    )

    trainer.fit(model, train_loader, val_loader)

    print(f"\n  Evaluating Unseen Test Data for Fold {fold}...")
    test_results = trainer.test(model, dataloaders=test_loader, ckpt_path='best', verbose=False)

    f_acc  = test_results[0]['test_acc']
    f_f1   = test_results[0]['test_f1']
    f_prec = test_results[0]['test_prec']
    f_rec  = test_results[0]['test_rec']

    print(f"  Fold {fold} Performance Metrics:")
    print(f"  >> Accuracy  : {f_acc:.4f} | F1-Score : {f_f1:.4f}")
    print(f"  >> Precision : {f_prec:.4f} | Recall   : {f_rec:.4f}")

    fold_metrics.append([f_acc, f_f1, f_prec, f_rec])

metrics_arr = np.array(fold_metrics)
means = np.mean(metrics_arr, axis=0)
stds = np.std(metrics_arr, axis=0)

print(f"\n{'='*65}")
print(f"  OVERALL 5-FOLD BENCHMARK RESULTS SUMMARY (EXP2: FULL PIPELINE)")
print(f"{'='*65}")
print(f"  Mean Accuracy  : {means[0]:.4f} ± {stds[0]:.4f}")
print(f"  Mean F1-Score  : {means[1]:.4f} ± {stds[1]:.4f}")
print(f"  Mean Precision : {means[2]:.4f} ± {stds[2]:.4f}")
print(f"  Mean Recall    : {means[3]:.4f} ± {stds[3]:.4f}")
print(f"{'='*65}")